# 04a · Preprocess images — once, so every epoch afterwards is GPU-bound

**Measured, not assumed:**

```
Drive random read : 49.4 img/s · 83 MB/s @32 threads  ->  2h05 per epoch, GPU idle
A6000 training    : ~55 min per epoch (FT), ~20 min (LP)
```

83 MB/s **is** Drive's ceiling — threading 1→8→16→32 saturated it. The problem is
not parallelism, it is that we ship **615 GB per epoch and then throw away 96% of
the pixels**: originals are up to 2544×3056, RAD-DINO eats 518².

| | now | after this notebook |
|---|---|---|
| bytes/epoch | 615 GB | **~18 GB** |
| read time/epoch | 2h05 | **~3 min** (local disk) |
| bottleneck | Drive | **GPU** ← where it belongs |
| GCS cost (requester-pays) | 615 GB × N epochs | 615 GB **once** |

One pass costs ~2h — the same as a single epoch today. It buys back ~40h over a
20-epoch run, and it is the difference between the training being feasible and not.

**Resumable by shard.** Colab will disconnect during a 2h pass; rerun and it skips
what exists.

**Defaults chosen here (flip them in C0):** grayscale (X-rays are single-channel;
RAD-DINO wants 3, so we repeat at load time — 3× smaller on disk), tar shards (a
few hundred ~50 MB files copy far faster than 365k small ones — the same latency
problem that makes Drive slow now).

**The target size is not guessed** — C1 reads it off RAD-DINO's own image processor.
Cropping a chest film wrongly cuts the apices or the costophrenic angles, which is
real pathology, so the geometry follows the model rather than my assumption.


## C0 · Config

In [ ]:
import os, io, sys, time, tarfile, json, random
import numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

CXR_BILLING = 'mimic-cxr-v210'          # requester-pays project for the image bucket
EHR_PROJECT = 'mimic-ehr-v310'          # BigQuery project -- NOT used here

from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
OUT_DIR  = os.path.join(DATA_DIR, 'cohort_v2')

# ---- source of the originals ----
# MEASURED, both at their own best thread count:
#     Drive @32t : 49.4 img/s · 83.0 MB/s
#     GCS   @32t : 51.4 img/s · 80.6 MB/s   (after fixing urllib3 pool 10 -> 256;
#                                            before that it read 28 img/s, all TLS churn)
# A tie. Two independent backends stopping at the same ~80 MB/s means the ceiling is
# BELOW both of them -- it is the Colab VM's network. So the source cannot buy speed,
# and Drive wins on cost: GCS is requester-pays and would bill 615 GB to mimic-cxr-v210
# for nothing.
SOURCE      = 'drive'                   # {'drive','gcs'} -- measured equal; drive is free
DRIVE_IMG   = '/content/drive/MyDrive/MIMIC_CXR/mimic-cxr-jpg-2.1.0.physionet.org/files'
GCS_BUCKET  = 'mimic-cxr-jpg-2.1.0.physionet.org'
GCS_POOL    = 256                       # urllib3 default is 10 -> 128 threads thrash it

# ---- split the pass across runtimes (optional) ----
# The ~80 MB/s ceiling is PER VM, so a second runtime gets its own ceiling: ~2x wall
# clock. Ranges must be disjoint; shards are rename-on-complete + skip-if-exists, so
# an overlap is wasteful but never corrupting.
#     runtime A:  SHARD_FROM, SHARD_TO = 0, None    -> then set TO = N_SHARDS//2
#     runtime B:  SHARD_FROM, SHARD_TO = None, None -> then set FROM = N_SHARDS//2
SHARD_FROM  = None                      # None -> 0
SHARD_TO    = None                      # None -> N_SHARDS  (i.e. this runtime does all)

# ---- output ----
SHARD_DIR   = os.path.join(DATA_DIR, 'cxr_560_shards')   # ~18 GB, written once
os.makedirs(SHARD_DIR, exist_ok=True)
SHARD_SIZE  = 1000                      # images per shard -> ~400 shards of ~45 MB
GRAYSCALE   = True                      # 1 channel on disk; repeat to 3 at load
JPEG_Q      = 90
N_THREADS   = 32                        # 32 saturated Drive; more will not help

# STORE_SIZE is set in C1 from RAD-DINO's own processor (+ crop headroom)
CROP_HEADROOM = 1.08                    # store ~8% larger than the model input so
                                        # RandomResizedCrop has something to crop
print('SOURCE     :', SOURCE, '(measured: drive 83 MB/s ~= gcs 80.6 MB/s -> VM-capped; drive is free)')
print('SHARD_DIR  :', SHARD_DIR)
print('SHARD RANGE:', SHARD_FROM, '->', SHARD_TO, '(None = all; set both to split across 2 runtimes)')

## C1 · Ask RAD-DINO what geometry it wants — do not assume it

`microsoft/rad-dino` ships an image processor. Whatever it does (resize shortest
edge → center crop, or squash to square) is what the pretrained weights saw, and
deviating silently costs accuracy that looks like a modelling failure later.

We store at `STORE_SIZE = model_size × 1.08` so the training-time crop has room,
then C3 applies **the processor's own resize rule** at the larger size.

In [ ]:
from transformers import AutoImageProcessor
proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
print('RAD-DINO image processor:')
for k in ('do_resize','size','resample','do_center_crop','crop_size','do_rescale',
          'do_normalize','image_mean','image_std'):
    if hasattr(proc, k): print(f'   {k:<18} {getattr(proc, k)}')

# the model input edge, taken from the processor rather than from memory
MODEL_SIZE = (proc.crop_size['height'] if getattr(proc, 'do_center_crop', False)
              else (proc.size.get('shortest_edge') or proc.size.get('height')))
STORE_SIZE = int(round(MODEL_SIZE * CROP_HEADROOM))
print(f'\nMODEL_SIZE (what the ViT sees) : {MODEL_SIZE}')
print(f'STORE_SIZE (what we cache)     : {STORE_SIZE}   (+{CROP_HEADROOM:.0%} crop headroom)')
print(f'bytes/image (grayscale q{JPEG_Q}) : ~{STORE_SIZE*STORE_SIZE*0.18/1024:.0f} KB est')

SHORTEST_EDGE = bool(getattr(proc, 'do_center_crop', False)) or \
                'shortest_edge' in (proc.size or {})
print(f'\nresize rule: {"shortest edge -> center crop (aspect preserved)" if SHORTEST_EDGE else "squash to square"}')
print('   -> C3 reproduces this rule at STORE_SIZE.')

## C2 · Work list — only images this project will ever use

Cohort studies (`01`) × frontal/lateral images (`03`). Studies excluded by §A1 and
images that stayed `other` are never read, so the pass is smaller than all of
MIMIC-CXR.

In [ ]:
base = pd.read_parquet(os.path.join(OUT_DIR,'cohort_base.parquet'))[['study_id','split']]
vm   = pd.read_parquet(os.path.join(OUT_DIR,'view_map.parquet'))
work = (vm[vm.view_group.isin(['frontal','lateral'])]
          .merge(base, on='study_id', how='inner')
          [['dicom_id','study_id','subject_id','view_group','split']]
          .sort_values(['subject_id','study_id','dicom_id'])
          .reset_index(drop=True))
print(f'cohort studies      : {len(base):,}')
print(f'images to preprocess: {len(work):,}')
print(work.view_group.value_counts().to_frame('n')
        .assign(frac=lambda d:(d.n/len(work)).round(4)).to_string())
print('\nby split:'); print(work.split.value_counts().to_frame('n_images').to_string())

work['shard'] = np.arange(len(work)) // SHARD_SIZE
N_SHARDS = int(work.shard.max()) + 1
print(f'\nshards: {N_SHARDS} x {SHARD_SIZE} images')
work.to_parquet(os.path.join(SHARD_DIR,'_work_list.parquet'), index=False)

## C3 · The pass — resumable, shard by shard

Each shard is written to a `.tmp` and renamed only when complete, so a disconnect
mid-shard cannot leave a half-written tar that a rerun would mistake for done.

In [ ]:
if SOURCE == 'gcs':
    import requests.adapters, logging
    from google.cloud import storage
    logging.getLogger('urllib3').setLevel(logging.ERROR)
    _sc = storage.Client(project=CXR_BILLING)
    # WITHOUT this the client keeps a 10-connection pool: 32 threads then spend their
    # time discarding and re-handshaking TLS, and throughput DROPS as you add threads.
    _ad = requests.adapters.HTTPAdapter(pool_connections=GCS_POOL,
                                        pool_maxsize=GCS_POOL, max_retries=3)
    _sc._http.mount('https://', _ad); _sc._http.mount('http://', _ad)
    _bkt = _sc.bucket(GCS_BUCKET, user_project=CXR_BILLING)
    print(f'gcs client ready, pool={GCS_POOL}')

def _read_bytes(sub, st, dic):
    s = str(int(sub)); rel = f'p{s[:2]}/p{s}/s{int(st)}/{dic}.jpg'
    if SOURCE == 'gcs':
        return _bkt.blob('files/' + rel).download_as_bytes()
    with open(os.path.join(DRIVE_IMG, rel), 'rb') as f:
        return f.read()

def _prep(raw):
    """Reproduce RAD-DINO's resize rule at STORE_SIZE."""
    im = Image.open(io.BytesIO(raw))
    im = im.convert('L') if GRAYSCALE else im.convert('RGB')
    if SHORTEST_EDGE:
        w, h = im.size; s = STORE_SIZE / min(w, h)
        im = im.resize((max(1, round(w*s)), max(1, round(h*s))), Image.BICUBIC)
        w, h = im.size; l, t = (w-STORE_SIZE)//2, (h-STORE_SIZE)//2
        im = im.crop((l, t, l+STORE_SIZE, t+STORE_SIZE))
    else:
        im = im.resize((STORE_SIZE, STORE_SIZE), Image.BICUBIC)
    buf = io.BytesIO(); im.save(buf, 'JPEG', quality=JPEG_Q, optimize=True)
    return buf.getvalue()

def _one(r):
    try:
        return r.dicom_id, _prep(_read_bytes(r.subject_id, r.study_id, r.dicom_id)), None
    except Exception as e:
        return r.dicom_id, None, f'{type(e).__name__}: {e}'

def build_shard(i):
    final = os.path.join(SHARD_DIR, f'shard_{i:05d}.tar')
    if os.path.exists(final): return 'skip', 0, []
    tmp  = final + '.tmp'                       # rename-on-complete: a killed run
    rows = work[work.shard == i]                # never leaves a valid-looking tar
    errs = []
    with tarfile.open(tmp, 'w') as tar, ThreadPoolExecutor(N_THREADS) as ex:
        for dic, blob, err in ex.map(_one, [r for r in rows.itertuples()]):
            if err: errs.append((dic, err)); continue
            info = tarfile.TarInfo(f'{dic}.jpg'); info.size = len(blob)
            tar.addfile(info, io.BytesIO(blob))
    os.replace(tmp, final)
    return 'done', len(rows) - len(errs), errs

LO = 0 if SHARD_FROM is None else int(SHARD_FROM)
HI = N_SHARDS if SHARD_TO is None else int(SHARD_TO)
assert 0 <= LO < HI <= N_SHARDS, f'bad range [{LO},{HI}) for {N_SHARDS} shards'
have = len([f for f in os.listdir(SHARD_DIR) if f.endswith('.tar')])
print(f'this runtime builds shards [{LO}, {HI})  of {N_SHARDS}   '
      f'({have} already on disk, from any runtime)')

t0, n_ok, all_errs, n_built = time.time(), 0, [], 0
for i in range(LO, HI):
    st, n, errs = build_shard(i)
    n_ok += n; all_errs += errs
    if st == 'done':
        n_built += 1
        el   = time.time() - t0
        rate = n_ok / max(el, 1e-9)
        left = (HI - i - 1) * SHARD_SIZE / max(rate, 1e-9)
        print(f'  shard {i+1:>4} (range {LO}-{HI})  {rate:5.1f} img/s  '
              f'elapsed {el/60:5.1f}m  eta {left/3600:5.2f}h  errors {len(all_errs)}', end='\r')
print(f'\n\ndone. shards built here: {n_built} | images written: {n_ok:,} | '
      f'errors: {len(all_errs)}')
if all_errs:
    pd.DataFrame(all_errs, columns=['dicom_id','error']).to_csv(
        os.path.join(SHARD_DIR,'_errors.csv'), index=False)
    print('first errors:', all_errs[:5])

## C4 · Verify the shards before trusting them

A tar that opens is not a tar that is correct. Check the count, decode a sample,
confirm the geometry matches `STORE_SIZE`, and measure the read speed we actually
bought.

In [ ]:
tars = sorted(f for f in os.listdir(SHARD_DIR) if f.endswith('.tar'))
tot = sum(os.path.getsize(os.path.join(SHARD_DIR,f)) for f in tars)
print(f'shards: {len(tars)} / {N_SHARDS}')
print(f'total : {tot/1e9:.2f} GB   (originals were ~615 GB -> {615/max(tot/1e9,1e-9):.0f}x smaller)')
if tot/1e9 > 50:
    print('!! >50 GB: this may not fit Colab local disk in 04b. Lower JPEG_Q or STORE_SIZE.')

missing = [i for i in range(N_SHARDS)
           if not os.path.exists(os.path.join(SHARD_DIR, f'shard_{i:05d}.tar'))]
if missing:
    print(f'\n!! {len(missing)} shards missing, e.g. {missing[:8]}')
    print(f'   contiguous gap: [{min(missing)}, {max(missing)+1})')
    print('   -> if you split across runtimes, the other one still owes this range;')
    print('      otherwise set SHARD_FROM/SHARD_TO to it and re-run C3.')
assert not missing, f'{len(missing)} shards missing -> see above'
print('\nOK  all shards present')

names = []
for f in tars[:3] + tars[-2:]:
    with tarfile.open(os.path.join(SHARD_DIR,f)) as t: names += t.getnames()
print(f'\nsampled {len(names)} members from 5 shards')

with tarfile.open(os.path.join(SHARD_DIR, tars[0])) as t:
    ms = t.getmembers()[:8]
    for m in ms[:4]:
        im = Image.open(io.BytesIO(t.extractfile(m).read()))
        print(f'   {m.name:<45} {im.size} {im.mode} {m.size/1024:5.1f} KB')
    sizes = []
    for m in ms:
        im = Image.open(io.BytesIO(t.extractfile(m).read())); sizes.append(im.size)
assert all(s == (STORE_SIZE, STORE_SIZE) for s in sizes), f'geometry drift: {set(sizes)}'
print(f'\nOK  every sampled image is {STORE_SIZE}x{STORE_SIZE}')

t0 = time.time(); n = 0
for f in tars[:4]:
    with tarfile.open(os.path.join(SHARD_DIR,f)) as t:
        for m in t.getmembers():
            Image.open(io.BytesIO(t.extractfile(m).read())).load(); n += 1
dt = time.time() - t0
print(f'\nsequential read+decode from Drive: {n/dt:.0f} img/s '
      f'-> {len(work)/(n/dt)/60:.1f} min per epoch')
print('   (random small-file read was 49 img/s = 125 min/epoch)')
print('   copy the shards to LOCAL disk in 04b -- Drive is still not where you train from.')
print('\nNEXT: 04b_raddino_lpft.ipynb')